# 🔧 Phase 3: Supervised Fine-Tuning with QLoRA

We fine-tune Qwen2.5-7B-Instruct using QLoRA (4-bit quantization + LoRA adapters).
Training is tracked in W&B. We evaluate on the held-out test set at the end.

**Expected training time on A100 40GB:** ~30–45 minutes
**VRAM usage:** ~18–22 GB

In [1]:
# ── Global config ─────────────────────────────────────────────────────────────
BASE_MODEL     = "Qwen/Qwen2.5-7B-Instruct"
SFT_OUTPUT_DIR = "./outputs/sft-qlora-checkpoint"
WANDB_PROJECT  = "project4-lora-dpo"
MAX_SEQ_LEN    = 1024
SEED           = 42
import os, json
os.environ["HF_TOKEN"]     = "s/hf_[A-Za-z0-9]*/hf_XXXXXXXXXXXXXXXXXXXXXXXX/g"
os.environ["WANDB_API_KEY"] = "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"
os.makedirs(SFT_OUTPUT_DIR, exist_ok=True)

In [2]:
# ── Load dataset ──────────────────────────────────────────────────────────────
import json
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer_temp = AutoTokenizer.from_pretrained(BASE_MODEL)

with open('./data/train_sft.json') as f:
    train_raw = json.load(f)
with open('./data/test_sft.json') as f:
    test_raw = json.load(f)

def format_to_text(example):
    """Convert messages list → single string using chat template."""
    return {
        "text": tokenizer_temp.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

train_dataset = Dataset.from_list([{"messages": ex["messages"]} for ex in train_raw])
test_dataset  = Dataset.from_list([{"messages": ex["messages"]} for ex in test_raw])

train_dataset = train_dataset.map(format_to_text)
test_dataset  = test_dataset.map(format_to_text)

print(f'Train: {len(train_dataset)} | Test: {len(test_dataset)}')
print('Sample text preview:')
print(train_dataset[0]["text"][:200])

Map:   0%|          | 0/1687 [00:00<?, ? examples/s]

Map:   0%|          | 0/188 [00:00<?, ? examples/s]

Train: 1687 | Test: 188
Sample text preview:
<|im_start|>system
You are a precise JSON extraction assistant. 
Given unstructured text, extract the requested fields and return ONLY valid JSON.
Do not include any explanation, markdown, or extra te


In [3]:
# ── Load tokenizer & model with QLoRA config ─────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

print('✅ Base model loaded in 4-bit')

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Base model loaded in 4-bit


In [4]:
# ── LoRA configuration ────────────────────────────────────────────────────────
# These are the key hyperparameters you can tune
lora_config = LoraConfig(
    r=16,                   # Rank — higher = more capacity, more VRAM
    lora_alpha=32,          # Scaling factor (usually 2x rank)
    lora_dropout=0.05,      # Dropout for regularization
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[        # Which layers to apply LoRA to
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# You should see something like: trainable params: ~40M (0.5%) of 7B total

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [5]:
# ── Training arguments ────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig
import wandb

wandb.init(project=WANDB_PROJECT, name="sft-qlora-run1")

sft_config = SFTConfig(
    output_dir=SFT_OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,     # Effective batch size = 4 * 4 = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_seq_length=MAX_SEQ_LEN,
    fp16=False,
    bf16=True,                          # A100 supports bf16 natively
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="wandb",
    seed=SEED,
    dataset_text_field="text",
)

print('✅ Training config ready')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: vishaalsai29 (vishaalsai29-unc-charlotte) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Training config ready


In [6]:
# ── Initialize and run trainer ────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
)

print('🚀 Starting SFT training...')
print('   Watch your W&B dashboard for the loss curve!')
trainer.train()

# Save final model
trainer.save_model(SFT_OUTPUT_DIR)
tokenizer.save_pretrained(SFT_OUTPUT_DIR)
print(f'✅ SFT model saved to {SFT_OUTPUT_DIR}')

Map:   0%|          | 0/1687 [00:00<?, ? examples/s]

Map:   0%|          | 0/188 [00:00<?, ? examples/s]

🚀 Starting SFT training...
   Watch your W&B dashboard for the loss curve!


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(),

Step,Training Loss,Validation Loss
50,0.039500,0.039484
100,0.030800,0.030783
150,0.029100,0.028032
200,0.028800,0.028391
250,0.027800,0.027395
300,0.026800,0.027450


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu

✅ SFT model saved to ./outputs/sft-qlora-checkpoint


In [7]:
# ── Evaluate SFT model on test set ────────────────────────────────────────────
# Same eval logic as baseline notebook
from peft import PeftModel
from tqdm import tqdm
import torch

# Reload SFT model cleanly for eval
sft_model = model
sft_model.eval()

SYSTEM_PROMPT = """You are a precise JSON extraction assistant. 
Given unstructured text, extract the requested fields and return ONLY valid JSON.
Do not include any explanation, markdown, or extra text. Output only the JSON object."""

def run_inference(instruction, input_text, model, tokenizer, max_new_tokens=256):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{instruction}\n\nText: {input_text}"}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 temperature=0.1, do_sample=True,
                                 pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def is_valid_json(text):
    try: json.loads(text.strip()); return True
    except: return False

def exact_match(pred, gt):
    try: return json.loads(pred.strip()) == json.loads(gt.strip())
    except: return False

def field_coverage(pred, gt):
    try:
        p, g = json.loads(pred.strip()), json.loads(gt.strip())
        return sum(1 for k, v in g.items() if p.get(k) == v) / len(g)
    except: return 0.0

with open('./data/test_sft.json') as f:
    test_data = json.load(f)

results_sft = []
for ex in tqdm(test_data[:100], desc="Evaluating SFT model"):
    pred = run_inference(ex['instruction'], ex['input_text'], sft_model, tokenizer)
    results_sft.append({
        'valid_json': is_valid_json(pred),
        'exact_match': exact_match(pred, ex['output']),
        'field_coverage': field_coverage(pred, ex['output']),
    })

n = len(results_sft)
sft_metrics = {
    'model': 'SFT Model (QLoRA fine-tuned)',
    'json_validity_rate': sum(r['valid_json'] for r in results_sft) / n,
    'exact_match_accuracy': sum(r['exact_match'] for r in results_sft) / n,
    'avg_field_coverage': sum(r['field_coverage'] for r in results_sft) / n,
    'n_samples': n
}

with open('./data/sft_metrics.json', 'w') as f:
    json.dump(sft_metrics, f, indent=2)

# Load baseline for comparison
with open('./data/baseline_metrics.json') as f:
    baseline = json.load(f)

print('\n' + '='*65)
print('📊 COMPARISON: Base Model vs SFT Model')
print('='*65)
print(f'{"Metric":<35} {"Base":>10} {"SFT":>10} {"Delta":>10}')
print('-'*65)
for metric in ['json_validity_rate', 'exact_match_accuracy', 'avg_field_coverage']:
    base_val = baseline[metric]
    sft_val  = sft_metrics[metric]
    delta    = sft_val - base_val
    print(f'{metric:<35} {base_val:>10.1%} {sft_val:>10.1%} {delta:>+10.1%}')
print('✅ Saved to sft_metrics.json')
wandb.finish()

Evaluating SFT model: 100%|██████████| 100/100 [04:30<00:00,  2.71s/it]


📊 COMPARISON: Base Model vs SFT Model
Metric                                    Base        SFT      Delta
-----------------------------------------------------------------
json_validity_rate                      100.0%     100.0%      +0.0%
exact_match_accuracy                     39.0%     100.0%     +61.0%
avg_field_coverage                       80.2%     100.0%     +19.8%
✅ Saved to sft_metrics.json


eval/loss,█▃▁▂▁▁
eval/runtime,▃▁█▆▅▅
eval/samples_per_second,▆█▁▃▄▄
eval/steps_per_second,▆█▁▃▃▃
train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇████
train/grad_norm,█▄▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▅█████▇▇▇▇▆▆▆▅▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁
train/loss,█▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
eval/loss,0.02745
eval/runtime,13.1473
